# QTL experiment quality control

This notebook assembles a single, review-focused table for F2 fish from the QTL experiments. It flags unusual samples using five quality-control metrics while keeping shoaling index as an outcome for visualization and correlation analysis.

Important behavior:

- QC flags are advisory. No fish is removed automatically.
- Every zMAD reference distribution pools all retained F2 fish across all retained experiments.
- Shoaling index is displayed but never contributes to the QC flag.
- Every QC metric uses the same selected frames. Choose either a continuous raw-recording window or an episode-based window in the settings.
- All functions used by this analysis are defined in this notebook so the workflow can be read from top to bottom.

## 1. Settings

Edit this cell before running the notebook. The experiment dates are inclusive. Set either date to None to disable that side of the date filter.

`QC_WINDOW_METHOD` controls every QC metric. Continuous mode uses raw-recording minutes and includes every episode type. Episode mode uses the canonical summary `inDishTime` clock and keeps only `QC_EPISODE` rows whose starts are strictly inside `EPISODE_WINDOW_MIN`.

In [ ]:
from pathlib import Path
import glob
import hashlib
import json
import os
import re
import warnings

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Locate the repository whether the notebook is launched from the repository root,
# Analyses, or Analyses/QTLmapping.
REPO_DIR = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'models').is_dir() and (candidate / 'functions').is_dir():
        REPO_DIR = candidate.resolve()
        break
if REPO_DIR is None:
    raise FileNotFoundError('Could not locate the jlsocialbehavior repository from the current directory.')
os.chdir(REPO_DIR)

from functions.arena_use import calculate_arena_use_evenness

# Input locations used by the current QTL workflow.
PROCESSING_SETTINGS_CSV = (
    '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use/'
    'temp/shoaling_selection/ShoalQTL_temp_processing/processingSettings.csv'
)
META_FILE = (
    '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/07_Data/'
    'Carlos/Behavior/QTL/MetaData_QTL.xlsx'
)
PROCESSING_DIR = str(Path(PROCESSING_SETTINGS_CSV).parent)
OUTPUT_DIR = (
    '//nasdcsr.unil.ch/RECHERCHE/FAC/FBM/CIG/jlarsch/default/D2c/03_Common_Use/'
    'temp/shoaling_selection/ShoalQTL_temp_output/qtl_quality_control'
)

# User-editable cohort filters. Dates are inclusive; None disables a boundary.
EXPERIMENT_START_DATE = '2026-06-01'
EXPERIMENT_END_DATE = None

# Apply one time-selection method to calculate every QC metric. Allowed values are
# 'continuous' and 'episode'. Continuous mode is the default.
QC_WINDOW_METHOD = 'continuous'
CONTINUOUS_WINDOW_MIN = (60, 120)

# Episode mode keeps matching episode starts strictly inside this window.
EPISODE_WINDOW_MIN = (80, 240)
QC_EPISODE = '02k20f'
QC_GENOTYPE = 'F2'

# Robust-z threshold used for both low and high tails.
ZMAD_THRESHOLD = 4
# Choose 'global' for one pooled reference, or 'selection_family' for
# separate Sel1 and Sel3 references. Other lines retain the pooled reference.
QC_THRESHOLD_MODE = 'selection_family'

# Raw-trajectory and arena-evenness settings copied from the trajectory-grid workflow.
EXPECTED_N_FISH = 35
FPS = 30
EPISODE_DURATION_MIN = 5
TRAJECTORY_FRAME_STRIDE = 30
SPACE_EVENNESS_BINS = 80

# Output controls. Caches avoid rereading large trajectory blocks on later runs.
USE_TRAJECTORY_CACHE = True
SAVE_OUTPUTS = True
FIGURE_DPI = 180
QC_POINTPLOT_HEIGHT = 7.5
FINAL_SI_PLOT_HEIGHT = 10.0
CACHE_DIR = str(Path(OUTPUT_DIR) / 'cache')

# One editable, colorblind-friendly code is used consistently in every QC figure.
QC_METRICS = {
    'speed': {
        'column': 'avgSpeed',
        'label': 'Average speed',
        'unit': 'mm/s',
        'color': '#D55E00',
        'abbreviation': 'speed',
    },
    'thigmotaxis': {
        'column': 'thigmoIndex',
        'label': 'Thigmotaxis distance',
        'unit': 'mm from arena centre',
        'color': '#0072B2',
        'abbreviation': 'thig',
    },
    'space_evenness': {
        'column': 'spaceEvenness',
        'label': 'Space-use evenness',
        'unit': 'normalized Shannon evenness',
        'color': '#009E73',
        'abbreviation': 'even',
    },
    'animal_size': {
        'column': 'anSize',
        'label': 'Animal size',
        'unit': 'mm',
        'color': '#CC79A7',
        'abbreviation': 'size',
    },
    'bout_duration': {
        'column': 'boutDur',
        'label': 'Bout duration',
        'unit': 's',
        'color': '#E6AB02',
        'abbreviation': 'bout',
    },
}

# Set a metric to (lower, upper) to enforce visible pointplot limits. Leave
# it as None to use Matplotlib's automatic limits from points, SD bars, and
# pooled zMAD reference lines. Example: 'speed': (0, 14).
QC_POINTPLOT_Y_LIMITS = {
    'speed': (0, 15),
    'thigmotaxis': (0, 35),
    'space_evenness': (0.3, 0.9),
    'animal_size': None,
    'bout_duration': (0,17),
}

# Marker shape and size are also metric-specific so multiple flags remain visible
# when their colored outlines are overlaid on one SI point.
QC_MARKERS = {
    'speed': ('o', 92),
    'thigmotaxis': ('s', 112),
    'space_evenness': ('D', 132),
    'animal_size': ('^', 152),
    'bout_duration': ('v', 172),
}

UNFLAGGED_COLOR = '#8C8C8C'
SUMMARY_COLOR = '#202020'

sns.set_theme(style='ticks', context='notebook')

### Metric definitions

The four summary metrics and shoaling index come from the canonical siSummary files written by models/experiment.py. Their rows are selected with the active `QC_WINDOW_METHOD`; space-use evenness is calculated from the exact matching raw frames.

- Average speed is avgSpeed in mm/s.
- Thigmotaxis is thigmoIndex, the mean radial distance from the arena centre in mm.
- Animal size is anSize in mm.
- boutDur retains the repository's historical name. It is the median interval between detected bout starts in seconds.
- Shoaling index is si and is used only as an outcome.
- Space-use evenness is normalized Shannon evenness over the fixed grid inside the expected arena radius. Every fish uses the same arena-bin denominator; values range from 0 for concentrated occupancy toward 1 for uniform occupancy.

## 2. Validate settings and prepare output locations

These small functions make invalid dates, window methods, or analysis windows fail before any large files are read. Continuous boundaries must align with the canonical five-minute summary blocks so every metric covers exactly the same frames.

In [ ]:
def parse_optional_date(value, setting_name):
    """Parse a YYYY-MM-DD setting, returning None when that filter is disabled."""
    if value is None:
        return None
    try:
        return pd.to_datetime(value, format='%Y-%m-%d', errors='raise').normalize()
    except (TypeError, ValueError) as exc:
        raise ValueError(f'{setting_name} must be None or a valid YYYY-MM-DD date.') from exc


def validate_minute_window(value, setting_name):
    """Validate and normalize a two-value minute window."""
    if (
        not isinstance(value, (tuple, list))
        or len(value) != 2
        or not all(np.isfinite(value))
    ):
        raise ValueError(f'{setting_name} must contain two finite minute values.')

    window_start, window_stop = map(float, value)
    if window_start < 0 or window_stop <= window_start:
        raise ValueError(f'{setting_name} must satisfy 0 <= start < stop.')
    return window_start, window_stop


def validate_window_method(value):
    """Normalize the user-editable QC window-method switch."""
    method = str(value).strip().casefold()
    if method not in {'continuous', 'episode'}:
        raise ValueError("QC_WINDOW_METHOD must be either 'continuous' or 'episode'.")
    return method


def validate_threshold_mode(value):
    """Normalize the user-editable zMAD reference-distribution switch."""
    mode = str(value).strip().casefold()
    if mode not in {'global', 'selection_family'}:
        raise ValueError(
            "QC_THRESHOLD_MODE must be either 'global' or 'selection_family'."
        )
    return mode


def validate_analysis_settings():
    """Validate all settings that affect cohort selection or QC calculations."""
    start_date = parse_optional_date(EXPERIMENT_START_DATE, 'EXPERIMENT_START_DATE')
    end_date = parse_optional_date(EXPERIMENT_END_DATE, 'EXPERIMENT_END_DATE')

    if start_date is not None and end_date is not None and start_date > end_date:
        raise ValueError('EXPERIMENT_START_DATE must not be later than EXPERIMENT_END_DATE.')

    window_method = validate_window_method(QC_WINDOW_METHOD)
    threshold_mode = validate_threshold_mode(QC_THRESHOLD_MODE)
    continuous_window = validate_minute_window(
        CONTINUOUS_WINDOW_MIN,
        'CONTINUOUS_WINDOW_MIN',
    )
    episode_window = validate_minute_window(
        EPISODE_WINDOW_MIN,
        'EPISODE_WINDOW_MIN',
    )

    if not np.isfinite(ZMAD_THRESHOLD) or ZMAD_THRESHOLD <= 0:
        raise ValueError('ZMAD_THRESHOLD must be a positive finite number.')
    if TRAJECTORY_FRAME_STRIDE < 1:
        raise ValueError('TRAJECTORY_FRAME_STRIDE must be at least 1.')
    if SPACE_EVENNESS_BINS < 2:
        raise ValueError('SPACE_EVENNESS_BINS must be at least 2.')

    return (
        start_date, end_date, window_method, threshold_mode, continuous_window, episode_window
    )


(
    FILTER_START_DATE,
    FILTER_END_DATE,
    QC_WINDOW_METHOD,
    QC_THRESHOLD_MODE,
    CONTINUOUS_WINDOW_MIN,
    EPISODE_WINDOW_MIN,
) = validate_analysis_settings()

# Small checks keep the two accepted switch values and invalid-value behavior visible.
assert validate_window_method('continuous') == 'continuous'
assert validate_window_method('EPISODE') == 'episode'
assert validate_threshold_mode('selection_family') == 'selection_family'
try:
    validate_window_method('not-a-method')
except ValueError:
    pass
else:
    raise AssertionError('An unknown QC window method must be rejected.')

# Create directories only when the corresponding output behavior is enabled.
if SAVE_OUTPUTS:
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
if USE_TRAJECTORY_CACHE:
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print('Repository:', REPO_DIR)
print('Processing settings:', PROCESSING_SETTINGS_CSV)
print('Metadata:', META_FILE)
print('Date filter:', FILTER_START_DATE, 'through', FILTER_END_DATE)
print('QC window method:', QC_WINDOW_METHOD)
print('Continuous raw-recording window:', CONTINUOUS_WINDOW_MIN, 'minutes')
print('Episode-mode in-dish window:', EPISODE_WINDOW_MIN, 'minutes')
print('Episode-mode episode:', QC_EPISODE)
print('QC threshold mode:', QC_THRESHOLD_MODE)
print('zMAD threshold:', ZMAD_THRESHOLD)

## 3. Load and aggregate canonical episode summaries

The loader below preserves a stable experiment key, maps local indices to global QTLIDs, and stops if an expected F2 fish lacks any selected summary block. It creates one shared trajectory selection per experiment so summary metrics and space-use evenness always cover the same frames.

In [ ]:
SUMMARY_METRIC_COLUMNS = ['avgSpeed', 'thigmoIndex', 'anSize', 'boutDur', 'si']
SUMMARY_REQUIRED_COLUMNS = set(SUMMARY_METRIC_COLUMNS + ['episode', 'epStart', 'inDishTime'])


def safe_filename(value):
    """Convert a label to a conservative filename component."""
    return ''.join(
        char if char.isalnum() or char in '._-' else '_'
        for char in str(value).strip()
    )


def normalize_episode(value):
    """Match the episode cleanup used by the QTL mapping notebook."""
    return str(value).strip().replace('_', '')


def parse_animal_numbers(value):
    """Parse experiment rosters such as '101:135', '101 102', or '101,102'."""
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return np.array([], dtype=int)
    if ':' in text:
        start_id, stop_id = text.split(':', 1)
        return np.arange(int(float(start_id)), int(float(stop_id)) + 1, dtype=int)
    return np.asarray([int(float(item)) for item in re.split(r'[\s,]+', text) if item], dtype=int)


def animal_numbers_for_experiment(row):
    """Return the ordered AllAn.anNr roster represented by one experiment row."""
    for column in ('anIDAll', 'anNr'):
        if column in row.index and pd.notna(row[column]):
            animal_numbers = parse_animal_numbers(row[column])
            if animal_numbers.size:
                return animal_numbers
    raise ValueError(f'No animal IDs were found for experiment {row.get("folder", row.name)}.')


def find_summary_file(txt_path):
    """Require exactly one canonical five-minute summary for a trajectory file."""
    txt_stem = Path(str(txt_path)).stem
    pattern = os.path.join(PROCESSING_DIR, txt_stem + '*siSummary_epi5.0.csv')
    matches = sorted(glob.glob(pattern))
    if len(matches) != 1:
        raise FileNotFoundError(
            f'Expected exactly one summary for {txt_stem}; found {len(matches)} using {pattern}'
        )
    return matches[0]


def prepare_animal_metadata(animal_metadata):
    """Validate and normalize the metadata columns used for QTL sample labels."""
    required = {'anNr', 'QTLID', 'line', 'genotype'}
    missing = required - set(animal_metadata.columns)
    if missing:
        raise ValueError('AllAn metadata is missing columns: ' + ', '.join(sorted(missing)))

    metadata = animal_metadata[['anNr', 'QTLID', 'line', 'genotype']].copy()
    metadata['anNr'] = pd.to_numeric(metadata['anNr'], errors='coerce')
    metadata['QTLID'] = metadata['QTLID'].astype('string').str.strip().replace('', pd.NA)
    metadata['genotype'] = metadata['genotype'].astype('string').str.strip()
    metadata['line'] = metadata['line'].astype('string').str.strip()

    if metadata['anNr'].isna().any():
        raise ValueError('AllAn contains missing or non-numeric anNr values.')
    metadata['anNr'] = metadata['anNr'].astype(int)
    duplicate_numbers = metadata.loc[metadata['anNr'].duplicated(keep=False), 'anNr'].unique()
    if duplicate_numbers.size:
        preview = ', '.join(map(str, duplicate_numbers[:10]))
        raise ValueError(f'AllAn contains duplicate anNr values: {preview}')

    # QTLID is the global identifier used by this F2 QC analysis. Validate it
    # only within F2 because non-F2 metadata rows legitimately reuse some IDs.
    f2_mask = metadata['genotype'].str.casefold().eq(QC_GENOTYPE.casefold())
    f2_metadata = metadata.loc[f2_mask]
    if f2_metadata['QTLID'].isna().any():
        raise ValueError('AllAn contains F2 rows with missing QTLID values.')
    duplicate_qtlids = f2_metadata.loc[
        f2_metadata['QTLID'].duplicated(keep=False), 'QTLID'
    ].unique()
    if duplicate_qtlids.size:
        preview = ', '.join(map(str, duplicate_qtlids[:10]))
        raise ValueError(f'AllAn contains duplicate F2 QTLID values: {preview}')

    return metadata


def map_summary_animal_numbers(summary, experiment_row):
    """Map zero-based summary animalIndex values to the experiment's AllAn.anNr roster."""
    if 'animalIndex' not in summary.columns:
        raise ValueError('Summary is missing the required animalIndex column.')

    # The canonical summary's animalID currently repeats 0..34 for every
    # experiment, so it must not be used as a global metadata identifier.
    animal_numbers = animal_numbers_for_experiment(experiment_row)
    local_index = pd.to_numeric(summary['animalIndex'], errors='coerce')
    valid = local_index.notna() & local_index.between(0, len(animal_numbers) - 1)
    mapped_numbers = pd.Series(np.nan, index=summary.index, dtype=float)
    mapped_numbers.loc[valid] = animal_numbers[local_index.loc[valid].astype(int)]

    if mapped_numbers.isna().any():
        raise ValueError('Some summary animalIndex values could not be mapped to AllAn.anNr.')
    return mapped_numbers.astype(int)


# Regression check: identical local indices in different experiments must map
# through their own rosters and therefore resolve to different global QTLIDs.
_synthetic_metadata = prepare_animal_metadata(pd.DataFrame({
    'anNr': [101, 102, 201, 202],
    'QTLID': ['Q1_A1', 'Q1_A2', 'Q2_A1', 'Q2_A2'],
    'line': ['line_a', 'line_a', 'line_b', 'line_b'],
    'genotype': ['F2', 'F2', 'F2', 'F2'],
}))
_synthetic_summary = pd.DataFrame({'animalIndex': [0, 1], 'animalID': [0, 1]})
_synthetic_qtlids = []
for _synthetic_roster in ('101:102', '201:202'):
    _synthetic_numbers = map_summary_animal_numbers(
        _synthetic_summary,
        pd.Series({'anNr': _synthetic_roster, 'folder': 'synthetic'}),
    )
    _synthetic_qtlids.append(
        _synthetic_metadata.set_index('anNr').loc[_synthetic_numbers, 'QTLID'].tolist()
    )
assert _synthetic_qtlids == [['Q1_A1', 'Q1_A2'], ['Q2_A1', 'Q2_A2']]


def select_experiment_rows(processing_settings):
    """Apply the inclusive, user-editable experiment date filter."""
    required = {'txtPath', 'date', 'folder'}
    missing = required - set(processing_settings.columns)
    if missing:
        raise ValueError('Processing settings are missing columns: ' + ', '.join(sorted(missing)))
    if processing_settings.index.duplicated().any():
        raise ValueError('Processing settings contain duplicate row indices.')

    selected = processing_settings.copy()
    selected['_experiment_date'] = pd.to_datetime(selected['date'], dayfirst=True, errors='coerce').dt.normalize()
    invalid_dates = selected['_experiment_date'].isna()
    if invalid_dates.any():
        preview = ', '.join(selected.loc[invalid_dates, 'date'].astype(str).unique()[:5])
        raise ValueError('Could not parse experiment dates: ' + preview)

    keep = pd.Series(True, index=selected.index)
    if FILTER_START_DATE is not None:
        keep &= selected['_experiment_date'] >= FILTER_START_DATE
    if FILTER_END_DATE is not None:
        keep &= selected['_experiment_date'] <= FILTER_END_DATE

    selected = selected.loc[keep].copy()
    if selected.empty:
        raise ValueError('No experiments remain after applying the date filter.')
    selected['experiment_key'] = selected.index.map(str)
    return selected


def select_qc_window_rows(
    summary,
    experiment_row,
    window_method,
    continuous_window,
    episode_window,
    qc_episode,
):
    """Select summary rows and matching raw-frame segments for one method."""
    block_duration_min = float(experiment_row.get('epiDur', EPISODE_DURATION_MIN))
    if not np.isfinite(block_duration_min) or block_duration_min <= 0:
        raise ValueError('epiDur must be a positive finite number of minutes.')
    block_frames = int(round(block_duration_min * 60 * FPS))

    # Each epStart must describe exactly one canonical block, even though the
    # summary repeats that block once per fish.
    block_columns = ['epStart', 'inDishTime', 'episode_clean']
    block_table = summary[block_columns].drop_duplicates().sort_values('epStart')
    if block_table['epStart'].duplicated().any():
        raise ValueError('One epStart has conflicting inDishTime or episode labels.')

    if window_method == 'continuous':
        window_start, window_stop = continuous_window
        start_frame_float = window_start * 60 * FPS
        stop_frame_float = window_stop * 60 * FPS
        start_frame = int(round(start_frame_float))
        stop_frame = int(round(stop_frame_float))
        if not np.isclose(start_frame_float, start_frame) or not np.isclose(
            stop_frame_float, stop_frame
        ):
            raise ValueError('Continuous-window boundaries must resolve to whole frames.')
        if start_frame % block_frames or stop_frame % block_frames:
            raise ValueError(
                'CONTINUOUS_WINDOW_MIN boundaries must align with canonical '
                f'{block_duration_min:g}-minute summary blocks.'
            )

        expected_starts = np.arange(start_frame, stop_frame, block_frames, dtype=int)
        available_starts = block_table['epStart'].astype(int).to_numpy()
        missing_starts = expected_starts[~np.isin(expected_starts, available_starts)]
        if missing_starts.size:
            preview = ', '.join(map(str, missing_starts[:6]))
            raise ValueError(
                'Continuous window is not completely tiled by summary blocks; '
                + 'missing epStart frame(s): '
                + preview
            )
        selected_rows = summary[summary['epStart'].isin(expected_starts)].copy()
        segments = ((start_frame, stop_frame),)
        selected_starts = expected_starts
        window_label = f'continuous_{window_start:g}-{window_stop:g}_raw_min'
    elif window_method == 'episode':
        window_start, window_stop = episode_window
        selected_rows = summary[
            summary['episode_clean'].eq(normalize_episode(qc_episode))
            & summary['inDishTime'].gt(window_start)
            & summary['inDishTime'].lt(window_stop)
        ].copy()
        selected_starts = np.sort(selected_rows['epStart'].dropna().astype(int).unique())
        segments = tuple(
            (int(start), int(start + block_frames)) for start in selected_starts
        )
        window_label = (
            f'episode_{normalize_episode(qc_episode)}_'
            f'{window_start:g}-{window_stop:g}_in_dish_min'
        )
    else:
        raise ValueError(f'Unknown QC window method: {window_method}')

    if selected_rows.empty or len(selected_starts) == 0:
        raise ValueError(f'No summary blocks were selected for {window_label}.')

    selection = {
        'window_method': window_method,
        'window_label': window_label,
        'segments': segments,
        'selected_summary_blocks': int(len(selected_starts)),
        'analyzed_duration_min': float(
            sum(stop - start for start, stop in segments) / (FPS * 60)
        ),
    }
    return selected_rows, selection


# Transparent selection checks use the same alternating five-minute schedule as
# the QTL summaries. The default continuous window must select six exact blocks.
_synthetic_starts = np.arange(24, dtype=int) * EPISODE_DURATION_MIN * 60 * FPS
_synthetic_window_summary = pd.DataFrame({
    'epStart': _synthetic_starts,
    'inDishTime': 10 + _synthetic_starts / (FPS * 60),
    'episode_clean': np.where(np.arange(24) % 2 == 0, '01k01f', '02k20f'),
})
_continuous_rows, _continuous_selection = select_qc_window_rows(
    _synthetic_window_summary,
    pd.Series({'epiDur': EPISODE_DURATION_MIN}),
    'continuous',
    (60, 90),
    (80, 240),
    '02k20f',
)
assert _continuous_selection['segments'] == ((108000, 162000),)
assert _continuous_selection['selected_summary_blocks'] == 6
assert set(_continuous_rows['episode_clean']) == {'01k01f', '02k20f'}
assert int(30 * 60 * FPS / TRAJECTORY_FRAME_STRIDE) == 1800
_episode_rows, _episode_selection = select_qc_window_rows(
    _synthetic_window_summary,
    pd.Series({'epiDur': EPISODE_DURATION_MIN}),
    'episode',
    (60, 90),
    (80, 240),
    '02k20f',
)
assert _episode_rows['episode_clean'].eq('02k20f').all()
try:
    select_qc_window_rows(
        _synthetic_window_summary,
        pd.Series({'epiDur': EPISODE_DURATION_MIN}),
        'continuous',
        (62, 88),
        (80, 240),
        '02k20f',
    )
except ValueError:
    pass
else:
    raise AssertionError('Partial summary-block boundaries must be rejected.')


def load_qtl_summary_samples(processing_settings, animal_metadata):
    """Load summaries and return per-fish samples plus shared frame selections."""
    metadata = prepare_animal_metadata(animal_metadata)
    selected_experiments = select_experiment_rows(processing_settings)
    summary_parts = []
    selection_parts = []
    errors = []

    for _, experiment_row in selected_experiments.iterrows():
        experiment_key = experiment_row['experiment_key']
        experiment_name = str(experiment_row['folder'])
        try:
            expected_numbers = animal_numbers_for_experiment(experiment_row)
            expected_f2 = metadata[
                metadata['anNr'].isin(expected_numbers)
                & metadata['genotype'].str.casefold().eq(QC_GENOTYPE.casefold())
            ]

            summary_path = find_summary_file(experiment_row['txtPath'])
            summary = pd.read_csv(summary_path, index_col=0)
            missing = SUMMARY_REQUIRED_COLUMNS - set(summary.columns)
            if missing:
                raise ValueError('Summary is missing columns: ' + ', '.join(sorted(missing)))

            summary['anNr'] = map_summary_animal_numbers(summary, experiment_row)
            summary['episode_clean'] = summary['episode'].map(normalize_episode)
            summary['inDishTime'] = pd.to_numeric(summary['inDishTime'], errors='coerce')
            summary['epStart'] = pd.to_numeric(summary['epStart'], errors='coerce')

            # The same selection object drives summary aggregation and raw
            # trajectory loading, so every QC metric covers the same frames.
            selected_rows, selection = select_qc_window_rows(
                summary,
                experiment_row,
                QC_WINDOW_METHOD,
                CONTINUOUS_WINDOW_MIN,
                EPISODE_WINDOW_MIN,
                QC_EPISODE,
            )

            selected_rows = selected_rows.merge(
                metadata,
                on='anNr',
                how='left',
                validate='many_to_one',
            )
            selected_f2 = selected_rows[
                selected_rows['genotype'].str.casefold().eq(QC_GENOTYPE.casefold())
            ].copy()

            if expected_f2.shape[0] and selected_f2.empty:
                raise ValueError('No F2 rows remain after the metadata merge.')
            if selected_f2.duplicated(['anNr', 'epStart']).any():
                raise ValueError('Selected summary contains duplicate fish-block rows.')

            # Require every expected F2 fish to contribute every selected block.
            expected_block_count = selection['selected_summary_blocks']
            fish_block_counts = (
                selected_f2.groupby('anNr')['epStart']
                .nunique()
                .reindex(expected_f2['anNr'], fill_value=0)
            )
            incomplete_fish = fish_block_counts[fish_block_counts.ne(expected_block_count)]
            if not incomplete_fish.empty:
                preview = ', '.join(
                    f'{animal_number}: {count}/{expected_block_count}'
                    for animal_number, count in incomplete_fish.head(10).items()
                )
                raise ValueError('Incomplete selected summary coverage by anNr: ' + preview)

            for metric_column in SUMMARY_METRIC_COLUMNS:
                selected_f2[metric_column] = pd.to_numeric(
                    selected_f2[metric_column],
                    errors='coerce',
                )

            selected_f2['experiment_key'] = experiment_key
            selected_f2['experiment'] = experiment_name
            selected_f2['date'] = experiment_row['_experiment_date']
            setup_value = experiment_row.get('setup', np.nan)
            if pd.isna(setup_value):
                raise ValueError('Experiment setup is missing; it is required for plot labels.')
            setup_text = re.sub(r'^[sS]', '', str(setup_value).strip())
            if re.fullmatch(r'\d+\.0+', setup_text):
                setup_text = str(int(float(setup_text)))
            if not setup_text:
                raise ValueError('Experiment setup is empty; it is required for plot labels.')
            selected_f2['setup'] = setup_value
            selected_f2['lineSet'] = (
                selected_f2['line'].astype(str)
                + '_'
                + experiment_row['_experiment_date'].strftime('%d-%m-%Y')
            )
            selected_f2['experiment_label'] = (
                selected_f2['lineSet'] + '_s' + setup_text
            )

            group_columns = [
                'experiment_key',
                'experiment',
                'date',
                'line',
                'lineSet',
                'experiment_label',
                'setup',
                'anNr',
                'QTLID',
                'genotype',
            ]
            per_fish = (
                selected_f2.groupby(group_columns, dropna=False, sort=True)[SUMMARY_METRIC_COLUMNS]
                .mean()
                .reset_index()
            )
            summary_parts.append(per_fish)

            selection_parts.append({
                'experiment_key': experiment_key,
                'experiment': experiment_name,
                **selection,
            })
        except Exception as exc:
            errors.append(f'{experiment_name}: {exc}')

    if errors:
        raise RuntimeError(
            'Summary loading was incomplete; fix these experiments before calculating pooled QC:\n- '
            + '\n- '.join(errors)
        )
    if not summary_parts:
        raise ValueError('No F2 summary samples were loaded.')

    samples = pd.concat(summary_parts, ignore_index=True)
    trajectory_selections = pd.DataFrame(selection_parts)

    if samples.duplicated(['experiment_key', 'QTLID']).any():
        raise AssertionError('Per-fish aggregation did not produce unique experiment-animal rows.')

    # lineSet plus setup is concise but still distinguishes recordings made
    # from the same line on the same date. Require one unique label per experiment.
    experiment_labels = samples[
        ['experiment_key', 'experiment_label']
    ].drop_duplicates()
    if experiment_labels['experiment_key'].duplicated().any():
        raise ValueError('One experiment produced more than one lineSet/setup label.')
    if experiment_labels['experiment_label'].duplicated().any():
        duplicates = experiment_labels.loc[
            experiment_labels['experiment_label'].duplicated(keep=False),
            'experiment_label',
        ].unique()
        raise ValueError('lineSet/setup labels are not unique: ' + ', '.join(duplicates))

    if trajectory_selections['experiment_key'].duplicated().any():
        raise AssertionError('Trajectory selections are not unique by experiment.')

    return samples, trajectory_selections, selected_experiments, metadata

In [ ]:
# Read the two small index tables first, then discover the canonical summaries.
processing_settings = pd.read_csv(PROCESSING_SETTINGS_CSV, index_col=0)
animal_metadata_raw = pd.read_excel(META_FILE, sheet_name='AllAn')

summary_samples, trajectory_selections, selected_experiments, animal_metadata = (
    load_qtl_summary_samples(processing_settings, animal_metadata_raw)
)

# Compare the loaded sample count with the F2 animals named by every selected
# experiment roster. A mismatch indicates an identifier or summary-coverage problem.
expected_sample_total = sum(
    int(
        (
            animal_metadata['anNr'].isin(animal_numbers_for_experiment(experiment_row))
            & animal_metadata['genotype'].str.casefold().eq(QC_GENOTYPE.casefold())
        ).sum()
    )
    for _, experiment_row in selected_experiments.iterrows()
)
assert summary_samples.shape[0] == expected_sample_total, (
    f'Loaded {summary_samples.shape[0]} samples, but metadata expects {expected_sample_total}.'
)

print('Selected experiments:', selected_experiments.shape[0])
print('Expected F2 experiment-animal samples:', expected_sample_total)
print('Loaded F2 experiment-animal samples:', summary_samples.shape[0])
print('Active QC time-selection method:', QC_WINDOW_METHOD)
display(
    trajectory_selections[
        [
            'experiment',
            'window_method',
            'window_label',
            'selected_summary_blocks',
            'analyzed_duration_min',
        ]
    ]
)
display(
    summary_samples[
        ['experiment_label', 'QTLID', 'anNr', 'genotype', *SUMMARY_METRIC_COLUMNS]
    ].head()
)

In [ ]:
summary_samples

## 4. Compute space-use evenness from the shared frame selection

The trajectory reader loads only x/y columns from the frame segments selected above. Continuous mode reads one exact start-inclusive, stop-exclusive raw range; episode mode reads and concatenates the selected episode blocks. Coordinates are centred and converted to millimetres using the same ROI, background-scale, and pxPmm fallbacks as the trajectory-grid notebook.

The shared arena-use helper builds a fixed grid spanning the expected arena diameter, counts selected trajectory frames in each bin to form per-fish probability maps, and normalizes Shannon entropy by every bin whose centre lies inside the expected arena radius. The notebook consumes the returned scores without rebuilding any part of the probability distribution.

An experiment-level cache is safe to reuse only when its raw-file modification time, window method, exact frame segments, stride, fish count, FPS, and arena scaling settings all match. Arena-use grids and probabilities are always recalculated by the shared helper from the cached trajectories.

In [ ]:
def resolve_trajectory_scale(experiment_row, n_fish):
    """Return per-fish crop centres, pixels/mm, arena radius, and scale source."""
    txt_path = str(experiment_row['txtPath'])
    start_dir = os.path.dirname(txt_path)
    arena_diameter_mm = float(experiment_row.get('arenaDiameter_mm', 70))

    roi_matches = sorted(glob.glob(os.path.join(start_dir, 'ROIdef*')))
    if roi_matches:
        roi_path = roi_matches[0]
        try:
            rois = np.atleast_2d(np.loadtxt(roi_path))
        except Exception:
            rois = np.atleast_2d(np.loadtxt(roi_path, delimiter=','))
        if rois.shape[0] < n_fish:
            raise ValueError(f'ROI file has {rois.shape[0]} rows but {n_fish} fish are expected.')
        radii_px = rois[:n_fish, -1].astype(float)
        px_per_mm = 2 * np.nanmean(radii_px) / arena_diameter_mm
        centres_px = radii_px + 2
        return centres_px, px_per_mm, arena_diameter_mm / 2, roi_path

    scale_matches = []
    for pattern in ('bgMed_scale*', '*bgMed.csv'):
        scale_matches.extend(sorted(glob.glob(os.path.join(start_dir, pattern))))
    if scale_matches:
        scale_path = scale_matches[0]
        rois = np.atleast_2d(np.loadtxt(scale_path, skiprows=1, delimiter=','))
        if rois.shape[0] < n_fish:
            raise ValueError(f'Scale file has {rois.shape[0]} rows but {n_fish} fish are expected.')
        radius_column = 3 if rois.shape[1] > 3 else -1
        radii_px = rois[:n_fish, radius_column].astype(float)
        px_per_mm = 2 * np.nanmean(radii_px) / arena_diameter_mm
        centres_px = radii_px + 2
        return centres_px, px_per_mm, arena_diameter_mm / 2, scale_path

    if 'pxPmm' in experiment_row.index and pd.notna(experiment_row['pxPmm']):
        warnings.warn(
            'No ROI/scale file found; using pxPmm and leaving coordinates uncentred for '
            + str(experiment_row.get('folder', experiment_row.name))
        )
        return np.zeros(n_fish), float(experiment_row['pxPmm']), arena_diameter_mm / 2, 'pxPmm column'

    raise FileNotFoundError('No ROI, background scale, or pxPmm value found beside ' + txt_path)


def trajectory_cache_path(experiment_row, selection, n_fish):
    """Build a cache filename from inputs that define the cached trajectories."""
    txt_path = str(experiment_row['txtPath'])
    try:
        raw_mtime = os.path.getmtime(txt_path)
    except OSError:
        raw_mtime = None

    cache_key = {
        'txt_path': txt_path,
        'mtime': raw_mtime,
        'window_method': str(selection['window_method']),
        'window_label': str(selection['window_label']),
        'segments': [list(segment) for segment in selection['segments']],
        'frame_stride': int(TRAJECTORY_FRAME_STRIDE),
        'fps': int(FPS),
        'n_fish': int(n_fish),
        'arena_diameter_mm': float(experiment_row.get('arenaDiameter_mm', 70)),
        'selection_schema': 'unified-qc-window-v2',
        'identity_schema': 'anNr-roster-to-QTLID-v1',
    }
    digest = hashlib.sha1(
        json.dumps(cache_key, sort_keys=True, default=str).encode('utf-8')
    ).hexdigest()[:12]
    label = safe_filename(f'{experiment_row.get("folder", Path(txt_path).stem)}_{digest}')
    return Path(CACHE_DIR) / f'{label}.npz', cache_key


def load_selected_trajectory_segments(experiment_row, selection):
    """Load and scale x/y data for the shared selected frame segments."""
    animal_numbers = animal_numbers_for_experiment(experiment_row)
    n_fish = len(animal_numbers)
    if n_fish != EXPECTED_N_FISH:
        raise ValueError(f'Expected {EXPECTED_N_FISH} animal IDs, found {n_fish}.')

    cache_path, cache_key = trajectory_cache_path(experiment_row, selection, n_fish)
    if USE_TRAJECTORY_CACHE and cache_path.exists():
        cached = np.load(cache_path, allow_pickle=False)
        return (
            cached['xy_mm'],
            cached['animal_numbers'].astype(int),
            float(cached['arena_radius_mm']),
            json.loads(str(cached['meta_json'])),
        )

    txt_path = str(experiment_row['txtPath'])
    centres_px, px_per_mm, arena_radius_mm, scale_source = resolve_trajectory_scale(
        experiment_row,
        n_fish,
    )
    if not np.isfinite(px_per_mm) or px_per_mm <= 0:
        raise ValueError(f'Invalid pixel scaling for {txt_path}: {px_per_mm}')

    segments = tuple((int(start), int(stop)) for start, stop in selection['segments'])
    if not segments:
        raise ValueError('The shared trajectory selection contains no frame segments.')

    raw_segments = []
    if txt_path.lower().endswith('.npy'):
        raw_array = np.load(txt_path, mmap_mode='r')
        for start_frame, requested_stop_frame in segments:
            stop_frame = min(requested_stop_frame, raw_array.shape[0])
            block_xy = np.asarray(
                raw_array[start_frame:stop_frame:TRAJECTORY_FRAME_STRIDE, :n_fish, :2],
                dtype=float,
            )
            if stop_frame < requested_stop_frame:
                warnings.warn(
                    f'{experiment_row.get("folder", experiment_row.name)} segment at '
                    f'{start_frame} ends at frame {stop_frame}, before requested '
                    f'frame {requested_stop_frame}.'
                )
            raw_segments.append(block_xy)
    else:
        first_line = pd.read_csv(txt_path, header=None, nrows=1, sep=':')
        first_value = str(first_line.values[0][0])
        first_character = first_value[0] if first_value else ''
        if first_character == '(':
            raise ValueError('Old Bonsai tuple format is not supported by the fast QC trajectory reader.')

        xy_columns = [column for fish in range(n_fish) for column in (fish * 3, fish * 3 + 1)]
        header_rows = 1 if first_character == 'X' else 0

        for start_frame, stop_frame in segments:
            segment_frames = stop_frame - start_frame
            raw_table = pd.read_csv(
                txt_path,
                header=None,
                sep=r'\s+',
                usecols=xy_columns,
                skiprows=start_frame + header_rows,
                nrows=segment_frames,
            ).to_numpy(dtype=float)
            if raw_table.shape[0] < segment_frames:
                warnings.warn(
                    f'{experiment_row.get("folder", experiment_row.name)} segment at '
                    f'{start_frame} contains {raw_table.shape[0]} of '
                    f'{segment_frames} requested frames.'
                )
            raw_table = raw_table[::TRAJECTORY_FRAME_STRIDE]
            raw_segments.append(raw_table.reshape(raw_table.shape[0], n_fish, 2))

    if not raw_segments or sum(segment.shape[0] for segment in raw_segments) == 0:
        raise ValueError('No trajectory samples were read from the shared frame selection.')

    raw_xy = np.concatenate(raw_segments, axis=0)
    xy_mm = (raw_xy - centres_px[np.newaxis, :, np.newaxis]) / px_per_mm
    cache_metadata = {
        'cache_key': cache_key,
        'scale_source': str(scale_source),
        'window_method': str(selection['window_method']),
        'window_label': str(selection['window_label']),
        'sampled_frames': int(xy_mm.shape[0]),
    }

    if USE_TRAJECTORY_CACHE:
        np.savez_compressed(
            cache_path,
            xy_mm=xy_mm,
            animal_numbers=animal_numbers,
            arena_radius_mm=np.asarray(arena_radius_mm),
            meta_json=np.asarray(json.dumps(cache_metadata, sort_keys=True, default=str)),
        )

    return xy_mm, animal_numbers, arena_radius_mm, cache_metadata


def compute_space_evenness(xy_mm, animal_numbers, arena_radius_mm):
    """Calculate expected-arena evenness with the shared trajectory helper."""
    arena_use = calculate_arena_use_evenness(
        xy_mm,
        arena_radius_mm=arena_radius_mm,
        bin_count=SPACE_EVENNESS_BINS,
    )
    if len(animal_numbers) != len(arena_use.evenness):
        raise ValueError('Animal-number and trajectory fish counts do not match.')
    return pd.DataFrame({
        'anNr': np.asarray(animal_numbers, dtype=int),
        'spaceEvenness': arena_use.evenness,
        'trajectory_valid_samples': arena_use.valid_in_arena_samples,
        'evenness_error': '',
    })


def collect_space_evenness(summary_sample_table, trajectory_selections, experiments):
    """Calculate evenness experiment by experiment while retaining readable failures."""
    results = []

    for _, experiment_row in experiments.iterrows():
        experiment_key = experiment_row['experiment_key']
        expected_samples = summary_sample_table[
            summary_sample_table['experiment_key'].eq(experiment_key)
        ]
        if expected_samples.empty:
            continue

        matching_selections = trajectory_selections[
            trajectory_selections['experiment_key'].eq(experiment_key)
        ]

        try:
            if len(matching_selections) != 1:
                raise ValueError(
                    f'Expected one shared trajectory selection; found {len(matching_selections)}.'
                )
            selection = matching_selections.iloc[0]
            xy_mm, animal_numbers, arena_radius_mm, _ = load_selected_trajectory_segments(
                experiment_row,
                selection,
            )
            experiment_evenness = compute_space_evenness(
                xy_mm,
                animal_numbers,
                arena_radius_mm,
            )
            # Raw trajectory columns are experiment-local slots. Resolve their
            # anNr values to the same global QTLIDs used by the summary table.
            experiment_evenness = experiment_evenness.merge(
                expected_samples[['anNr', 'QTLID']],
                on='anNr',
                how='inner',
                validate='one_to_one',
            ).drop(columns='anNr')
            experiment_evenness['experiment_key'] = experiment_key
            results.append(experiment_evenness)
        except Exception as exc:
            message = str(exc)
            warnings.warn(
                f'Evenness could not be calculated for {experiment_row.get("folder", experiment_key)}: '
                + message
            )
            results.append(pd.DataFrame({
                'experiment_key': experiment_key,
                'QTLID': expected_samples['QTLID'].astype('string'),
                'spaceEvenness': np.nan,
                'trajectory_valid_samples': 0,
                'evenness_error': message,
            }))

    evenness = pd.concat(results, ignore_index=True)
    if evenness.duplicated(['experiment_key', 'QTLID']).any():
        raise AssertionError('Evenness results contain duplicate experiment-animal rows.')
    return evenness

In [ ]:
# Calculate trajectory-derived evenness, then preserve every summary sample with
# a one-to-one left merge. Failed trajectory loads therefore remain visible as NaN.
space_evenness = collect_space_evenness(
    summary_samples,
    trajectory_selections,
    selected_experiments,
)

qc_samples = summary_samples.merge(
    space_evenness,
    on=['experiment_key', 'QTLID'],
    how='left',
    validate='one_to_one',
)

if qc_samples.duplicated(['experiment_key', 'QTLID']).any():
    raise AssertionError('QC table contains duplicate experiment-animal rows.')

print('Samples with space-use evenness:', qc_samples['spaceEvenness'].notna().sum())
display(
    qc_samples[
        [
            'experiment_label',
            'QTLID',
            'anNr',
            'spaceEvenness',
            'trajectory_valid_samples',
            'evenness_error',
        ]
    ].head()
)

## 5. Calculate zMAD values and QC flags

`QC_THRESHOLD_MODE = 'global'` calculates each metric's median and MAD from every finite retained F2 value. Set it to `'selection_family'` to calculate independent references for Sel1 and Sel3; rows outside those families retain the global pooled reference.

A finite value is an outlier when the absolute zMAD is greater than or equal to the editable threshold. Missing values receive a separate missing flag. The combined review flag is the union of outliers and missing data.

In [ ]:
def robust_zmad(values):
    """Return median/MAD robust z-scores while handling missing and zero-MAD data."""
    numeric = pd.to_numeric(pd.Series(values), errors='coerce')
    median = numeric.median(skipna=True)
    mad = (numeric - median).abs().median(skipna=True)

    if pd.isna(mad) or mad == 0:
        zmad = pd.Series(
            np.where(numeric.notna(), 0.0, np.nan),
            index=numeric.index,
            dtype=float,
        )
    else:
        zmad = 0.6745 * (numeric - median) / mad

    return zmad, median, mad


def selection_family_for_threshold(value):
    """Return the QC threshold family for a metadata line, when defined."""
    normalized = str(value).strip().casefold()
    if normalized.startswith(('sel1', 'tlsel1')):
        return 'Sel1'
    if normalized.startswith(('sel3', 'tlsel3')):
        return 'Sel3'
    return None


def threshold_reference_groups(flagged):
    """Return reference rows and target rows for the active threshold mode."""
    all_rows = flagged.index
    if QC_THRESHOLD_MODE == 'global':
        flagged['qc_threshold_group'] = 'Global pooled'
        return [('Global pooled', all_rows, all_rows)]

    flagged['qc_threshold_group'] = flagged['line'].map(selection_family_for_threshold)
    flagged['qc_threshold_group'] = flagged['qc_threshold_group'].fillna('Global pooled')
    groups = []
    for family in ('Sel1', 'Sel3'):
        family_rows = flagged.index[flagged['qc_threshold_group'].eq(family)]
        if len(family_rows):
            groups.append((family, family_rows, family_rows))

    fallback_rows = flagged.index[flagged['qc_threshold_group'].eq('Global pooled')]
    if len(fallback_rows):
        groups.append(('Global pooled', all_rows, fallback_rows))
    return groups


def add_qc_flags(sample_table):
    """Add active-reference zMAD/missing flags, overall flags, and reasons."""
    flagged = sample_table.copy()
    reference_rows = []
    reference_groups = threshold_reference_groups(flagged)

    for metric_name, metric_info in QC_METRICS.items():
        value_column = metric_info['column']
        flagged[value_column] = pd.to_numeric(flagged[value_column], errors='coerce')
        zmad_column = f'{metric_name}_zmad'
        outlier_column = f'{metric_name}_outlier_flag'
        missing_column = f'{metric_name}_missing_flag'
        flagged[zmad_column] = np.nan

        for group_name, reference_rows_index, target_rows_index in reference_groups:
            zmad, median, mad = robust_zmad(flagged.loc[reference_rows_index, value_column])
            flagged.loc[target_rows_index, zmad_column] = zmad.loc[target_rows_index]

            if pd.notna(mad) and mad > 0:
                raw_delta = ZMAD_THRESHOLD * mad / 0.6745
                lower_threshold = median - raw_delta
                upper_threshold = median + raw_delta
            else:
                lower_threshold = np.nan
                upper_threshold = np.nan

            reference_rows.append({
                'metric': metric_name,
                'threshold_group': group_name,
                'threshold_mode': QC_THRESHOLD_MODE,
                'value_column': value_column,
                'median': median,
                'mad': mad,
                'lower_threshold': lower_threshold,
                'upper_threshold': upper_threshold,
                'finite_n': int(flagged.loc[reference_rows_index, value_column].notna().sum()),
                'zmad_threshold': ZMAD_THRESHOLD,
            })

        flagged[outlier_column] = flagged[zmad_column].abs().ge(ZMAD_THRESHOLD).fillna(False)
        flagged[missing_column] = flagged[value_column].isna()

    outlier_columns = [f'{metric}_outlier_flag' for metric in QC_METRICS]
    missing_columns = [f'{metric}_missing_flag' for metric in QC_METRICS]
    flagged['qc_outlier_flag'] = flagged[outlier_columns].any(axis=1)
    flagged['qc_missing_flag'] = flagged[missing_columns].any(axis=1)
    flagged['qc_review_flag'] = flagged['qc_outlier_flag'] | flagged['qc_missing_flag']

    def reason_text(row):
        reasons = []
        for metric_name, metric_info in QC_METRICS.items():
            if row[f'{metric_name}_outlier_flag']:
                reasons.append(metric_info['label'] + ' outlier')
            if row[f'{metric_name}_missing_flag']:
                reasons.append(metric_info['label'] + ' missing')
        return '; '.join(reasons)

    flagged['qc_reasons'] = flagged.apply(reason_text, axis=1)
    return flagged, pd.DataFrame(reference_rows)


# Small transparent checks document the intended QC-threshold behavior.
inclusive_check = pd.Series([-ZMAD_THRESHOLD, ZMAD_THRESHOLD - 0.01, ZMAD_THRESHOLD])
assert inclusive_check.abs().ge(ZMAD_THRESHOLD).tolist() == [True, False, True]

zero_mad_z, _, zero_mad = robust_zmad([4, 4, 4])
assert zero_mad == 0
assert np.allclose(zero_mad_z, 0)

qc_table, qc_reference = add_qc_flags(qc_samples)

print('Unique outlier fish:', int(qc_table['qc_outlier_flag'].sum()))
print('Samples requiring review, including missing data:', int(qc_table['qc_review_flag'].sum()))
display(qc_reference)

## 6. Coverage and flagged-sample tables

Coverage is shown before plotting so missing summaries or trajectories cannot be mistaken for clean samples. Outlier counts refer to unique experiment-animal rows. A fish failing multiple metrics is counted once in the unique total and once for every metric it failed.

In [ ]:
def expected_f2_count(experiment_row, metadata):
    """Count metadata F2 fish listed in one processing-settings row."""
    experiment_numbers = animal_numbers_for_experiment(experiment_row)
    return int(
        (
            metadata['anNr'].isin(experiment_numbers)
            & metadata['genotype'].str.casefold().eq(QC_GENOTYPE.casefold())
        ).sum()
    )


def build_experiment_coverage(qc_data, experiments, metadata):
    """Summarize expected, loaded, missing, and flagged fish per experiment."""
    rows = []
    metric_columns = [info['column'] for info in QC_METRICS.values()]

    for _, experiment_row in experiments.iterrows():
        experiment_key = experiment_row['experiment_key']
        experiment_data = qc_data[qc_data['experiment_key'].eq(experiment_key)]
        if experiment_data.empty:
            label = str(experiment_row.get('folder', experiment_key))
        else:
            label = experiment_data['experiment_label'].iloc[0]

        row = {
            'experiment_key': experiment_key,
            'experiment_label': label,
            'date': experiment_row['_experiment_date'],
            'expected_f2_fish': expected_f2_count(experiment_row, metadata),
            'loaded_summary_fish': int(experiment_data['QTLID'].nunique()),
            'evenness_available_fish': int(experiment_data['spaceEvenness'].notna().sum()),
            'any_metric_missing_fish': int(experiment_data[metric_columns].isna().any(axis=1).sum()),
            'unique_outlier_fish': int(experiment_data['qc_outlier_flag'].sum()),
            'unique_review_fish': int(experiment_data['qc_review_flag'].sum()),
        }
        for metric_name in QC_METRICS:
            row[f'{metric_name}_outliers'] = int(
                experiment_data[f'{metric_name}_outlier_flag'].sum()
            )
        rows.append(row)

    return pd.DataFrame(rows).sort_values(['date', 'experiment_label']).reset_index(drop=True)


coverage_table = build_experiment_coverage(
    qc_table,
    selected_experiments,
    animal_metadata,
)

flagged_columns = [
    'experiment_label',
    'date',
    'QTLID',
    'anNr',
    'line',
    'genotype',
    *[info['column'] for info in QC_METRICS.values()],
    'si',
    'qc_threshold_group',
    *[f'{metric}_zmad' for metric in QC_METRICS],
    'qc_outlier_flag',
    'qc_missing_flag',
    'qc_review_flag',
    'qc_reasons',
]
flagged_samples = (
    qc_table.loc[qc_table['qc_review_flag'], flagged_columns]
    .sort_values(['date', 'experiment_label', 'QTLID'])
    .reset_index(drop=True)
)

display(coverage_table)
display(flagged_samples)

if SAVE_OUTPUTS:
    qc_table.to_csv(Path(OUTPUT_DIR) / 'qtl_quality_control_metrics.csv', index=False)
    flagged_samples.to_csv(Path(OUTPUT_DIR) / 'qtl_quality_control_flagged.csv', index=False)
    coverage_table.to_csv(Path(OUTPUT_DIR) / 'qtl_quality_control_coverage.csv', index=False)
    qc_reference.to_csv(Path(OUTPUT_DIR) / 'qtl_quality_control_reference.csv', index=False)

## 7. Per-experiment plots for every QC metric

Each figure follows the standard QTL shoaling-index display: all fish are shown, and an overlaid pointplot shows the experiment mean and standard deviation. Global mode shows pooled raw-value cutoffs; selection-family mode shows separate Sel1 and Sel3 cutoffs. Metric-specific outliers are colored; they are not removed from the mean or SD.

In [ ]:
def experiment_plot_order(data):
    """Order experiment labels chronologically, retaining a stable secondary label sort."""
    return (
        data[['experiment_label', 'date']]
        .drop_duplicates()
        .sort_values(['date', 'experiment_label'])['experiment_label']
        .tolist()
    )


def stable_jitter_position(experiment_label, qtlid, experiment_order, width=0.16):
    """Create repeatable horizontal jitter from the experiment-animal identifier."""
    x_centre = experiment_order.index(experiment_label)
    token = f'{experiment_label}|{qtlid}'.encode('utf-8')
    integer = int(hashlib.sha1(token).hexdigest()[:8], 16)
    unit_value = integer / float(16 ** 8 - 1)
    return x_centre + (unit_value - 0.5) * 2 * width


def add_stable_plot_positions(data, experiment_order):
    """Attach deterministic x positions for QC annotation overlays."""
    positioned = data.copy()
    positioned['_plot_x'] = [
        stable_jitter_position(label, qtlid, experiment_order)
        for label, qtlid in zip(positioned['experiment_label'], positioned['QTLID'])
    ]
    return positioned


def save_figure(fig, stem):
    """Save both review-friendly PNG and vector PDF versions when enabled."""
    if not SAVE_OUTPUTS:
        return
    fig.savefig(Path(OUTPUT_DIR) / f'{stem}.png', dpi=FIGURE_DPI, bbox_inches='tight')
    fig.savefig(Path(OUTPUT_DIR) / f'{stem}.pdf', bbox_inches='tight')


def pointplot_y_limits(metric_name):
    """Validate one optional user-editable y-axis override."""
    unknown_metrics = set(QC_POINTPLOT_Y_LIMITS) - set(QC_METRICS)
    if unknown_metrics:
        raise KeyError(
            'QC_POINTPLOT_Y_LIMITS contains unknown metric key(s): '
            + ', '.join(sorted(unknown_metrics))
        )

    limits = QC_POINTPLOT_Y_LIMITS.get(metric_name)
    if limits is None:
        return None
    if (
        not isinstance(limits, (tuple, list))
        or len(limits) != 2
        or not all(np.isfinite(limits))
    ):
        raise ValueError(
            f'QC_POINTPLOT_Y_LIMITS[{metric_name!r}] must be None or '
            'a two-value finite (lower, upper) tuple.'
        )
    lower_limit, upper_limit = map(float, limits)
    if upper_limit <= lower_limit:
        raise ValueError(
            f'QC_POINTPLOT_Y_LIMITS[{metric_name!r}] must satisfy lower < upper.'
        )
    return lower_limit, upper_limit


def plot_metric_by_experiment(data, reference_table, metric_name):
    """Plot all fish plus mean/SD and active-reference cutoffs for one metric."""
    metric_info = QC_METRICS[metric_name]
    value_column = metric_info['column']
    outlier_column = f'{metric_name}_outlier_flag'
    order = experiment_plot_order(data)
    positioned = add_stable_plot_positions(data, order)
    finite = positioned[positioned[value_column].notna()].copy()

    fig_width = max(9, 0.8 * len(order) + 2)
    fig, ax = plt.subplots(
        figsize=(fig_width, QC_POINTPLOT_HEIGHT),
        dpi=FIGURE_DPI,
    )

    # Draw every fish at its deterministic position before highlighting outliers.
    ax.scatter(
        finite['_plot_x'],
        finite[value_column],
        s=34,
        color=UNFLAGGED_COLOR,
        alpha=0.55,
        edgecolor='white',
        linewidth=0.4,
        zorder=2,
        label='all fish',
    )
    metric_outliers = finite[finite[outlier_column]]
    ax.scatter(
        metric_outliers['_plot_x'],
        metric_outliers[value_column],
        s=58,
        color=metric_info['color'],
        edgecolor='white',
        linewidth=0.7,
        alpha=0.95,
        zorder=4,
        label=f'{metric_info["label"]} outlier',
    )

    # The pointplot includes every finite fish, including highlighted outliers.
    sns.pointplot(
        data=finite,
        x='experiment_label',
        y=value_column,
        order=order,
        color=SUMMARY_COLOR,
        errorbar='sd',
        linestyle='-',
        markers='D',
        capsize=0.12,
        ax=ax,
    )

    threshold_styles = {
        'Global pooled': {'color': metric_info['color'], 'linestyle': ':'},
        'Sel1': {'color': '#7B3294', 'linestyle': '--'},
        'Sel3': {'color': '#008837', 'linestyle': '-.'},
    }
    metric_references = reference_table.loc[reference_table['metric'].eq(metric_name)]
    for _, reference in metric_references.iterrows():
        group_name = reference['threshold_group']
        style = threshold_styles[group_name]
        for threshold_value, threshold_label in (
            (reference['lower_threshold'], f'-{reference["zmad_threshold"]:g} zMAD'),
            (reference['upper_threshold'], f'+{reference["zmad_threshold"]:g} zMAD'),
        ):
            if np.isfinite(threshold_value):
                ax.axhline(
                    threshold_value,
                    linewidth=1.6,
                    alpha=0.9,
                    label=(
                        f'{group_name} {threshold_label}: {threshold_value:.3g}'
                    ),
                    **style,
                )

    # Apply a user-requested range after every artist is drawn, so the chosen
    # limits govern points, SD bars, and active threshold lines together.
    y_limits = pointplot_y_limits(metric_name)
    if y_limits is not None:
        ax.set_ylim(*y_limits)

    ax.set_xlabel('Experiment')
    ax.set_ylabel(f'{metric_info["label"]} ({metric_info["unit"]})')
    ax.set_title(
        f'{metric_info["label"]} per experiment: all F2 fish, mean +/- SD'
    )
    ax.tick_params(axis='x', rotation=90)
    ax.legend(frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left')
    sns.despine(ax=ax)
    fig.tight_layout()
    return fig, ax


metric_figures = {}
for metric_name in QC_METRICS:
    metric_figures[metric_name] = plot_metric_by_experiment(
        qc_table,
        qc_reference,
        metric_name,
    )
    save_figure(
        metric_figures[metric_name][0],
        f'qtl_qc_{metric_name}_by_experiment',
    )

plt.show()

### Additional QC plots ordered by line and date

These figures retain the original QC display and statistics while grouping experiments by selection line and ordering recording dates chronologically within each line.

In [ ]:
def experiment_plot_order_by_line_and_date(data):
    """Order experiments case-insensitively by line, then date and label."""
    return (
        data[['experiment_label', 'line', 'date']]
        .drop_duplicates()
        .assign(_line_order=lambda frame: frame['line'].str.casefold())
        .sort_values(['_line_order', 'date', 'experiment_label'])
        ['experiment_label']
        .tolist()
    )


def median_mad_interval(values):
    """Return median +/- normal-scaled MAD for a finite numeric sample."""
    numeric = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    median = numeric.median()
    mad = (numeric - median).abs().median()
    scaled_mad = 1.4826 * mad
    return median - scaled_mad, median + scaled_mad


qc_line_palette = {
    'Sel1A': '#E29844',
    'Sel1B': '#E2551E',
    'Sel3A': '#163F97',
    'Sel3C': '#7555A9',
    'Sel3D': '#2C7938',
}
qc_family_palette = {
    'Sel1': '#CE9223',
    'Sel3': '#1B5699',
}
qc_panel_families = ['Sel1', 'Sel3']


def plot_metric_by_line_and_date(data, reference_table, metric_name):
    """Plot one QC metric in line/date-ordered Sel1 and Sel3 panels."""
    metric_info = QC_METRICS[metric_name]
    value_column = metric_info['column']
    outlier_column = f'{metric_name}_outlier_flag'

    panel_data = data.copy()
    panel_data['selection_family'] = panel_data['line'].map(
        selection_family_for_threshold
    )
    panel_data = panel_data.loc[
        panel_data['selection_family'].isin(qc_panel_families)
    ].copy()
    if panel_data.empty:
        raise ValueError('No Sel1 or Sel3 samples are available for the QC panels.')

    missing_line_colors = (
        set(panel_data['line'].dropna().unique()) - set(qc_line_palette)
    )
    if missing_line_colors:
        raise ValueError(
            'Missing QC panel colors for lines: '
            + ', '.join(sorted(missing_line_colors))
        )

    experiment_count = panel_data['experiment_label'].nunique()
    fig_width = max(13, 0.8 * experiment_count + 4)
    fig, axes = plt.subplots(
        1,
        2,
        figsize=(fig_width, QC_POINTPLOT_HEIGHT),
        dpi=FIGURE_DPI,
        sharey='row',
    )
    plotted_lines = []
    threshold_handles = []

    for ax, family in zip(axes, qc_panel_families):
        family_data = panel_data.loc[
            panel_data['selection_family'].eq(family)
        ].copy()
        if family_data.empty:
            raise ValueError(f'No {family} samples are available for the QC panels.')

        order = experiment_plot_order_by_line_and_date(family_data)
        positioned = add_stable_plot_positions(family_data, order)
        finite = positioned[positioned[value_column].notna()].copy()
        family_lines = (
            family_data[['line']]
            .drop_duplicates()
            .assign(_line_order=lambda frame: frame['line'].str.casefold())
            .sort_values('_line_order')['line']
            .tolist()
        )
        plotted_lines.extend(family_lines)

        ax.scatter(
            finite['_plot_x'],
            finite[value_column],
            s=34,
            c=finite['line'].map(qc_line_palette),
            alpha=0.55,
            edgecolor='white',
            linewidth=0.4,
            zorder=2,
        )
        metric_outliers = finite[finite[outlier_column]]
        ax.scatter(
            metric_outliers['_plot_x'],
            metric_outliers[value_column],
            s=76,
            facecolors='none',
            edgecolors=metric_info['color'],
            linewidth=1.6,
            zorder=4,
        )

        sns.pointplot(
            data=finite,
            x='experiment_label',
            y=value_column,
            hue='line',
            hue_order=family_lines,
            palette=qc_line_palette,
            order=order,
            dodge=False,
            estimator=np.median,
            errorbar=median_mad_interval,
            linestyle='-',
            markers='D',
            capsize=0.12,
            legend=False,
            ax=ax,
        )

        family_references = reference_table.loc[
            reference_table['metric'].eq(metric_name)
            & reference_table['threshold_group'].eq(family)
        ]
        for _, reference in family_references.iterrows():
            for threshold_value, threshold_label, linestyle in (
                (
                    reference['lower_threshold'],
                    f'-{reference["zmad_threshold"]:g} zMAD',
                    ':',
                ),
                (
                    reference['upper_threshold'],
                    f'+{reference["zmad_threshold"]:g} zMAD',
                    '--',
                ),
            ):
                if np.isfinite(threshold_value):
                    ax.axhline(
                        threshold_value,
                        color=qc_family_palette[family],
                        linestyle=linestyle,
                        linewidth=1.6,
                        alpha=0.9,
                    )
                    threshold_handles.append(
                        Line2D(
                            [0],
                            [0],
                            color=qc_family_palette[family],
                            linestyle=linestyle,
                            linewidth=1.6,
                            label=(
                                f'{family} {threshold_label}: '
                                f'{threshold_value:.3g}'
                            ),
                        )
                    )

        ax.set_xlabel('Experiment (line, then date)')
        ax.set_title(family, fontsize=16, fontweight='bold', pad=12)
        ax.tick_params(axis='x', rotation=90)
        sns.despine(ax=ax)

    y_limits = pointplot_y_limits(metric_name)
    if y_limits is not None:
        axes[0].set_ylim(*y_limits)

    axes[0].set_ylabel(f'{metric_info["label"]} ({metric_info["unit"]})')
    axes[1].set_ylabel('')
    fig.suptitle(
        f'{metric_info["label"]} per experiment: all F2 fish, median +/- normal-scaled MAD',
        fontweight='bold',
    )

    line_handles = [
        Line2D(
            [0],
            [0],
            marker='o',
            linestyle='none',
            markerfacecolor=qc_line_palette[line],
            markeredgecolor='white',
            markersize=7,
            label=line,
        )
        for line in plotted_lines
    ]
    outlier_handle = Line2D(
        [0],
        [0],
        marker='o',
        linestyle='none',
        markerfacecolor='none',
        markeredgecolor=metric_info['color'],
        markeredgewidth=1.6,
        markersize=8,
        label=f'{metric_info["label"]} outlier',
    )
    fig.legend(
        handles=[*line_handles, outlier_handle, *threshold_handles],
        title='Line and QC reference',
        frameon=False,
        bbox_to_anchor=(0.85, 0.92),
        loc='upper left',
    )
    fig.tight_layout(rect=[0, 0, 0.82, 0.95])
    fig.subplots_adjust(wspace=0.08)
    return fig, axes


line_date_metric_figures = {}
for metric_name in QC_METRICS:
    line_date_metric_figures[metric_name] = plot_metric_by_line_and_date(
        qc_table,
        qc_reference,
        metric_name,
    )
    save_figure(
        line_date_metric_figures[metric_name][0],
        f'qtl_qc_{metric_name}_by_line_and_date',
    )

plt.show()

## 8. QC-metric distributions by selection line

Each panel compares individual F2 fish across the Sel1 and Sel3 line families. Sel1 lines are shown first, followed by Sel3 lines; values from other line families are excluded from this focused comparison.

In [ ]:
def format_p_value(value):
    """Use compact scientific notation only for small p-values."""
    if not np.isfinite(value):
        return 'N/A'
    return f'{value:.2e}' if value < 0.001 else f'{value:.3f}'


def plot_si_correlation_panel(ax, data, metric_name, title):
    """Plot one complete-case QC metric/SI correlation without excluding flags."""
    metric_info = QC_METRICS[metric_name]
    value_column = metric_info['column']
    outlier_column = f'{metric_name}_outlier_flag'
    complete = data.dropna(subset=[value_column, 'si']).copy()
    metric_outliers = complete[complete[outlier_column]]

    ax.scatter(
        complete[value_column],
        complete['si'],
        s=34,
        color=UNFLAGGED_COLOR,
        alpha=0.55,
        edgecolor='white',
        linewidth=0.4,
        label='all complete fish',
    )
    ax.scatter(
        metric_outliers[value_column],
        metric_outliers['si'],
        s=58,
        color=metric_info['color'],
        alpha=0.95,
        edgecolor='white',
        linewidth=0.7,
        label=f'{metric_info["label"]} outlier',
    )

    if (
        len(complete) >= 3
        and complete[value_column].nunique() > 1
        and complete['si'].nunique() > 1
    ):
        pearson_r, p_value = stats.pearsonr(complete[value_column], complete['si'])
        slope, intercept = np.polyfit(complete[value_column], complete['si'], 1)
        x_line = np.linspace(complete[value_column].min(), complete[value_column].max(), 100)
        ax.plot(x_line, slope * x_line + intercept, color=SUMMARY_COLOR, linewidth=1.4)
    else:
        pearson_r, p_value = np.nan, np.nan

    ax.text(
        0.03,
        0.97,
        f'r = {pearson_r:.3f}\np = {format_p_value(p_value)}\nn = {len(complete)}',
        transform=ax.transAxes,
        ha='left',
        va='top',
        bbox={'boxstyle': 'round', 'facecolor': 'white', 'alpha': 0.85},
    )
    ax.axhline(0, color='0.5', linestyle=':', linewidth=1)
    ax.set_xlabel(f'{metric_info["label"]} ({metric_info["unit"]})')
    ax.set_ylabel('Shoaling index')
    ax.set_title(title)
    ax.legend(
        frameon=False,
        fontsize=8,
        loc='upper center',
        bbox_to_anchor=(0.5, -0.24),
        ncol=2,
        borderaxespad=0,
    )
    sns.despine(ax=ax)
    return complete, pearson_r, p_value


def plot_si_correlations(data):
    """Plot pooled QC metric/SI correlations without excluding flagged samples."""
    fig, axes = plt.subplots(3, 2, figsize=(12, 17), dpi=FIGURE_DPI)
    axes = axes.ravel()

    for ax, (metric_name, metric_info) in zip(axes, QC_METRICS.items()):
        plot_si_correlation_panel(
            ax, data, metric_name, f'SI versus {metric_info["label"].lower()}'
        )

    for unused_ax in axes[len(QC_METRICS):]:
        unused_ax.remove()

    fig.suptitle('Correlations of QC metrics with individual shoaling index', y=1.01)
    fig.tight_layout(h_pad=4.5)
    return fig, axes


def plot_si_correlations_by_selection_family(data):
    """Plot independent Sel1 and Sel3 correlations for every QC metric."""
    family_data = data.copy()
    family_data['_selection_family'] = family_data['line'].map(
        selection_family_for_threshold
    )
    families = ('Sel1', 'Sel3')
    fig, axes = plt.subplots(
        len(QC_METRICS),
        len(families),
        figsize=(12, 5 * len(QC_METRICS)),
        dpi=FIGURE_DPI,
        sharex='row',
        squeeze=False,
    )

    for row_index, (metric_name, metric_info) in enumerate(QC_METRICS.items()):
        for column_index, family in enumerate(families):
            subset = family_data[family_data['_selection_family'].eq(family)]
            plot_si_correlation_panel(
                axes[row_index, column_index],
                subset,
                metric_name,
                f'{family}: SI versus {metric_info["label"].lower()}',
            )

    fig.suptitle(
        'QC metric correlations with individual shoaling index by selection family',
        y=1.005,
    )
    fig.tight_layout(h_pad=4.5)
    return fig, axes


correlation_figure, _ = plot_si_correlations(qc_table)
save_figure(correlation_figure, 'qtl_qc_si_correlations')

family_correlation_figure, _ = plot_si_correlations_by_selection_family(qc_table)
save_figure(
    family_correlation_figure,
    'qtl_qc_si_correlations_by_selection_family',
)
plt.show()

## 10. Ranked zMAD heatmap

Rows are ranked by their largest absolute QC zMAD. This provides a compact view of single-metric and multi-metric deviations. The complete table remains available even when the figure suppresses row labels for a very large cohort.

In [ ]:
def plot_ranked_zmad_heatmap(data):
    """Plot sample-by-metric zMAD values ordered by maximum absolute deviation."""
    zmad_columns = [f'{metric_name}_zmad' for metric_name in QC_METRICS]
    heatmap_data = data.copy()
    heatmap_data['_max_abs_zmad'] = heatmap_data[zmad_columns].abs().max(axis=1)
    heatmap_data = heatmap_data.sort_values(
        ['_max_abs_zmad', 'date', 'experiment_label', 'QTLID'],
        ascending=[False, True, True, True],
    )
    heatmap_data['_sample_label'] = (
        heatmap_data['experiment_label'].astype(str)
        + ' | '
        + heatmap_data['QTLID'].astype(str)
    )

    matrix = heatmap_data.set_index('_sample_label')[zmad_columns]
    matrix.columns = [QC_METRICS[name]['label'] for name in QC_METRICS]

    figure_height = max(5, min(30, 0.22 * len(matrix)))
    fig, ax = plt.subplots(figsize=(9, figure_height), dpi=FIGURE_DPI)
    show_row_labels = len(matrix) <= 100

    sns.heatmap(
        matrix,
        cmap='vlag',
        center=0,
        vmin=-max(3, ZMAD_THRESHOLD),
        vmax=max(3, ZMAD_THRESHOLD),
        yticklabels=show_row_labels,
        cbar_kws={'label': f'{QC_THRESHOLD_MODE} zMAD'},
        ax=ax,
    )
    ax.set_xlabel('QC metric')
    ax.set_ylabel('Experiment and fish' if show_row_labels else 'Samples ranked by absolute zMAD')
    ax.set_title('QTL sample quality-control zMAD heatmap')
    fig.tight_layout()
    return fig, ax, heatmap_data


zmad_figure, _, ranked_qc_samples = plot_ranked_zmad_heatmap(qc_table)
save_figure(zmad_figure, 'qtl_qc_ranked_zmad_heatmap')
plt.show()

display(
    ranked_qc_samples.loc[
        ranked_qc_samples['qc_review_flag'],
        flagged_columns,
    ].reset_index(drop=True)
)

## 11. Final shoaling-index plot with metric-specific QC overlays

Every fish appears once as a gray filled point. Colored hollow markers are then overlaid at the same deterministic jitter position for each failed metric. A fish with multiple failures therefore displays multiple colors and marker shapes. Fish with missing animal size receive a distinct X marker without being reclassified as outliers.

Counts above each experiment report unique outlier fish, each metric-specific outlier count, and the separate count of fish with missing animal size.

In [ ]:
def plot_si_with_qc_outliers(data):
    """Recreate the SI-per-experiment view with multi-metric QC overlays and counts."""
    order = experiment_plot_order(data)
    positioned = add_stable_plot_positions(data, order)
    complete_si = positioned[positioned['si'].notna()].copy()

    fig_width = max(10, 0.85 * len(order) + 2)
    fig, ax = plt.subplots(
        figsize=(fig_width, FINAL_SI_PLOT_HEIGHT),
        dpi=FIGURE_DPI,
    )

    # The gray filled point is the one base point for every fish.
    ax.scatter(
        complete_si['_plot_x'],
        complete_si['si'],
        s=38,
        color=UNFLAGGED_COLOR,
        alpha=0.58,
        edgecolor='white',
        linewidth=0.45,
        zorder=2,
    )

    # Mean and SD include all complete SI values, regardless of QC status.
    sns.pointplot(
        data=complete_si,
        x='experiment_label',
        y='si',
        order=order,
        color=SUMMARY_COLOR,
        errorbar='sd',
        linestyle='-',
        markers='D',
        capsize=0.12,
        ax=ax,
    )

    # Each metric is an independent overlay. Different shapes and sizes keep
    # simultaneous flags visible instead of forcing one arbitrary priority.
    for metric_name, metric_info in QC_METRICS.items():
        marker, marker_size = QC_MARKERS[metric_name]
        metric_outliers = complete_si[complete_si[f'{metric_name}_outlier_flag']]
        ax.scatter(
            metric_outliers['_plot_x'],
            metric_outliers['si'],
            s=marker_size,
            marker=marker,
            facecolors='none',
            edgecolors=metric_info['color'],
            linewidths=1.7,
            zorder=5,
        )

    # Missing animal size requires review but remains distinct from an outlier.
    size_missing = complete_si[complete_si['animal_size_missing_flag']]
    ax.scatter(
        size_missing['_plot_x'],
        size_missing['si'],
        s=132,
        marker='X',
        color='#7A3D00',
        linewidths=1.2,
        zorder=6,
    )

    # Count unique outlier fish once, while retaining per-metric counts that can
    # overlap when one fish fails more than one metric.
    for x_position, experiment_label in enumerate(order):
        experiment_data = data[data['experiment_label'].eq(experiment_label)]
        unique_total = int(experiment_data['qc_outlier_flag'].sum())
        ax.text(
            x_position,
            1.24,
            f'out={unique_total}',
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='bottom',
            fontsize=8,
            fontweight='bold',
            color=SUMMARY_COLOR,
        )
        for count_index, (metric_name, metric_info) in enumerate(QC_METRICS.items()):
            metric_count = int(experiment_data[f'{metric_name}_outlier_flag'].sum())
            ax.text(
                x_position,
                1.205 - 0.035 * count_index,
                f'{metric_info["abbreviation"]}={metric_count}',
                transform=ax.get_xaxis_transform(),
                ha='center',
                va='bottom',
                fontsize=6.8,
                color=metric_info['color'],
            )
        size_missing_count = int(experiment_data['animal_size_missing_flag'].sum())
        ax.text(
            x_position,
            1.205 - 0.035 * len(QC_METRICS),
            f'size missing={size_missing_count}',
            transform=ax.get_xaxis_transform(),
            ha='center',
            va='bottom',
            fontsize=6.8,
            color='#7A3D00',
        )

    legend_handles = [
        Line2D(
            [0],
            [0],
            marker='o',
            linestyle='none',
            markerfacecolor=UNFLAGGED_COLOR,
            markeredgecolor='white',
            label='all fish',
            markersize=7,
        )
    ]
    for metric_name, metric_info in QC_METRICS.items():
        marker, _ = QC_MARKERS[metric_name]
        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker=marker,
                linestyle='none',
                markerfacecolor='none',
                markeredgecolor=metric_info['color'],
                markeredgewidth=1.7,
                label=metric_info['label'],
                markersize=9,
            )
        )
    legend_handles.append(
        Line2D(
            [0],
            [0],
            marker='X',
            linestyle='none',
            markerfacecolor='#7A3D00',
            markeredgecolor='#7A3D00',
            label='Animal size missing',
            markersize=8,
        )
    )

    ax.axhline(0, color='0.35', linestyle=':', linewidth=1)
    ax.set_xlabel('Experiment')
    ax.set_ylabel('Shoaling index')
    ax.set_ylim(-0.05, 0.5)
    ax.set_title(
        'Shoaling index per experiment with QC flags',
        y=1.33,
        pad=0,
    )
    ax.tick_params(axis='x', rotation=90)
    ax.legend(
        handles=legend_handles,
        title='QC overlay',
        frameon=False,
        bbox_to_anchor=(1.01, 1),
        loc='upper left',
    )
    sns.despine(ax=ax)
    fig.tight_layout()
    return fig, ax


final_si_figure, _ = plot_si_with_qc_outliers(qc_table)
save_figure(final_si_figure, 'qtl_qc_si_by_experiment_with_outliers')
plt.show()

## 12. Outlier counts and animal-size missingness by line

Each row reports the unique fish flagged either as a QC outlier or for missing animal size, together with the separate diagnostic counts. Fish with both flags are counted once in `Unique flagged fish (outlier or size missing)`. Metric counts are independent, so a fish that fails multiple QC metrics appears in multiple category columns but only once in `Unique outlier fish`.

In [ ]:
outlier_summary_data = qc_table.assign(
    outlier_or_animal_size_missing_flag=(
        qc_table['qc_outlier_flag'] | qc_table['animal_size_missing_flag']
    )
)

outlier_summary_columns = {
    'outlier_or_animal_size_missing_flag': (
        'Unique flagged fish (outlier or size missing)'
    ),
    'qc_outlier_flag': 'Unique outlier fish',
    'animal_size_missing_flag': 'Animal size missing',
    **{
        f'{metric_name}_outlier_flag': metric_info['label']
        for metric_name, metric_info in QC_METRICS.items()
    },
}

outlier_summary_by_line = (
    outlier_summary_data
    .groupby('line', dropna=False)[list(outlier_summary_columns)]
    .sum()
    .astype(int)
    .rename(columns=outlier_summary_columns)
    .sort_index(key=lambda values: values.astype(str).str.casefold())
)
outlier_summary_by_line.loc['All lines'] = (
    outlier_summary_data[list(outlier_summary_columns)]
    .sum()
    .astype(int)
    .rename(index=outlier_summary_columns)
)

display(outlier_summary_by_line)